In [ ]:
#Descarga de las imagenes, mover las imagenes al directorio actual de trabajo
import kagglehub

# Download latest version
path = kagglehub.dataset_download("udaypyarasani/faceforensics-preprocessed-face-dataset")

print("Path to dataset files:", path)

In [1]:
import os
import shutil
import random
from collections import defaultdict

# Configuración de carpetas origen y destino
BASE_DIR = "."  # Directorio actual
REAL_DIR = os.path.join(BASE_DIR, "Real")
FAKE_DIR = os.path.join(BASE_DIR, "Fake")

REAL_MINI_DIR = os.path.join(BASE_DIR, "Real_mini")
FAKE_MINI_DIR = os.path.join(BASE_DIR, "Fake_mini")

# Crear carpetas de destino si no existen
os.makedirs(REAL_MINI_DIR, exist_ok=True)
os.makedirs(FAKE_MINI_DIR, exist_ok=True)

# Extensiones de imagen soportadas
VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff')

# Lista auxiliar para acumular los registros
registros = []

# -------------------------------------------------------------
# 1. Procesar imágenes reales (250 aleatorias) -> Clasificación: 0
# -------------------------------------------------------------
if os.path.exists(REAL_DIR):
    real_files = [f for f in os.listdir(REAL_DIR) if f.lower().endswith(VALID_EXTENSIONS)]
    print(f"Total de imágenes encontradas en 'Real': {len(real_files)}")
    
    selected_real = random.sample(real_files, min(250, len(real_files)))
    
    for idx, filename in enumerate(selected_real, start=1):
        ext = os.path.splitext(filename)[1]
        src_path = os.path.join(REAL_DIR, filename)
        dst_filename = f"real_{idx:03d}{ext}"
        dst_path = os.path.join(REAL_MINI_DIR, dst_filename)
        
        # Copiar imagen a la nueva carpeta
        shutil.copy2(src_path, dst_path)
        
        # Ruta absoluta o URL del archivo
        file_url = os.path.abspath(dst_path)
        
        # Guardar tupla: (0, 'real', url_imagen)
        registros.append((0, 'real', file_url))
        
    print(f"✓ Copiadas {len(selected_real)} imágenes a 'Real_mini'\n")
else:
    print("❌ No se encontró la carpeta 'Real'. Revisa la ruta.\n")

# -------------------------------------------------------------
# 2. Procesar imágenes fake (150 de cada uno de los 5 tipos = 750) -> Clasificación: 1
# -------------------------------------------------------------
CATEGORIES = ["deepfakes", "face2face", "faceshifter", "faceswap", "neuraltextures"]

if os.path.exists(FAKE_DIR):
    fake_files = [f for f in os.listdir(FAKE_DIR) if f.lower().endswith(VALID_EXTENSIONS)]
    print(f"Total de imágenes encontradas en 'Fake': {len(fake_files)}")
    
    categorized_fakes = defaultdict(list)
    for filename in fake_files:
        filename_lower = filename.lower()
        for cat in CATEGORIES:
            if filename_lower.startswith(cat):
                categorized_fakes[cat].append(filename)
                break

    total_fake_copied = 0
    
    for cat in CATEGORIES:
        files = categorized_fakes[cat]
        print(f"Imágenes encontradas de '{cat}': {len(files)}")
        
        selected = random.sample(files, min(150, len(files)))
        
        for idx, filename in enumerate(selected, start=1):
            ext = os.path.splitext(filename)[1]
            src_path = os.path.join(FAKE_DIR, filename)
            dst_filename = f"{cat}_{idx:03d}{ext}"
            dst_path = os.path.join(FAKE_MINI_DIR, dst_filename)
            
            # Copiar imagen
            shutil.copy2(src_path, dst_path)
            
            # Ruta absoluta o URL del archivo
            file_url = os.path.abspath(dst_path)
            
            # Guardar tupla: (1, tipo_fake, url_imagen)
            registros.append((1, cat, file_url))
            
        total_fake_copied += len(selected)
        print(f"  └─ Copiadas {len(selected)} imágenes de '{cat}' a 'Fake_mini'")
        
    print(f"\n✓ Total de imágenes copiadas a 'Fake_mini': {total_fake_copied}")
else:
    print("❌ No se encontró la carpeta 'Fake'. Revisa la ruta.")

# -------------------------------------------------------------
# 3. Convertir a la Tupla Final de elementos
# -------------------------------------------------------------
dataset_tuple = tuple(registros)

print(f"\n¡Proceso completado! Total de registros en la tupla: {len(dataset_tuple)}")

Total de imágenes encontradas en 'Real': 5000
✓ Copiadas 250 imágenes a 'Real_mini'

Total de imágenes encontradas en 'Fake': 6000
Imágenes encontradas de 'deepfakes': 1200
  └─ Copiadas 150 imágenes de 'deepfakes' a 'Fake_mini'
Imágenes encontradas de 'face2face': 1200
  └─ Copiadas 150 imágenes de 'face2face' a 'Fake_mini'
Imágenes encontradas de 'faceshifter': 1200
  └─ Copiadas 150 imágenes de 'faceshifter' a 'Fake_mini'
Imágenes encontradas de 'faceswap': 1200
  └─ Copiadas 150 imágenes de 'faceswap' a 'Fake_mini'
Imágenes encontradas de 'neuraltextures': 1200
  └─ Copiadas 150 imágenes de 'neuraltextures' a 'Fake_mini'

✓ Total de imágenes copiadas a 'Fake_mini': 750

¡Proceso completado! Total de registros en la tupla: 1000


In [ ]:
import requests
import json
import os
import time

# Clave API de Hive
API_KEY = ''

headers = {
    'authorization': f'Bearer {API_KEY}',
}

# Lista donde se guardarán los resultados de los 3 modelos para cada imagen
# Formato: [[res_hive, None, None], [res_hive, None, None], ...]
respuestas_modelos = []

if 'dataset_tuple' in locals() and len(dataset_tuple) > 0:
    print(f"Iniciando evaluación con Hive para {len(dataset_tuple)} imágenes...\n")
    
    for idx, (clasificacion, tipo_fake, file_path) in enumerate(dataset_tuple, start=1):
        print(f"[{idx}/{len(dataset_tuple)}] Procesando: {os.path.basename(file_path)} ({tipo_fake})")
        
        try:
            # Abrir y enviar el archivo local en el cuerpo de la petición (multipart/form-data)
            with open(file_path, 'rb') as f:
                files = {'media': (os.path.basename(file_path), f, 'image/jpeg')}
                
                response = requests.post(
                    'https://api.thehive.ai/api/v3/hive/ai-generated-and-deepfake-content-detection',
                    headers=headers,
                    files=files
                )
                
                if response.status_code == 200:
                    res_hive = response.json()
                else:
                    res_hive = {"error_code": response.status_code, "message": response.text}
                    
        except Exception as e:
            res_hive = {"error": str(e)}
        
        # Guardar en la estructura de 3 posiciones (Hive, Modelo2=None, Modelo3=None)
        respuestas_modelos.append([res_hive, None, None])
        
        # Pausa ligera (0.1s) recomendada entre peticiones
        time.sleep(0.1)

    print(f"\n✓ Proceso finalizado. Se han evaluado {len(respuestas_modelos)} imágenes con Hive.")

    # Guardar los resultados en un archivo JSON local
    with open("respuestas_modelos.json", "w", encoding="utf-8") as out_file:
        json.dump(respuestas_modelos, out_file, ensure_ascii=False, indent=4)
    print("✓ Resultados guardados en 'respuestas_modelos.json'")

else:
    print("❌ Error: La tupla 'dataset_tuple' no existe o está vacía.")

Iniciando evaluación con Hive para 1000 imágenes...

[1/1000] Procesando: real_001.jpg (real)
[2/1000] Procesando: real_002.jpg (real)
[3/1000] Procesando: real_003.jpg (real)
[4/1000] Procesando: real_004.jpg (real)
[5/1000] Procesando: real_005.jpg (real)
[6/1000] Procesando: real_006.jpg (real)
[7/1000] Procesando: real_007.jpg (real)
[8/1000] Procesando: real_008.jpg (real)
[9/1000] Procesando: real_009.jpg (real)
[10/1000] Procesando: real_010.jpg (real)
[11/1000] Procesando: real_011.jpg (real)
[12/1000] Procesando: real_012.jpg (real)
[13/1000] Procesando: real_013.jpg (real)
[14/1000] Procesando: real_014.jpg (real)
[15/1000] Procesando: real_015.jpg (real)
[16/1000] Procesando: real_016.jpg (real)
[17/1000] Procesando: real_017.jpg (real)
[18/1000] Procesando: real_018.jpg (real)
[19/1000] Procesando: real_019.jpg (real)
[20/1000] Procesando: real_020.jpg (real)
[21/1000] Procesando: real_021.jpg (real)
[22/1000] Procesando: real_022.jpg (real)
[23/1000] Procesando: real_023.j

In [ ]:
import requests
import json
import os
import time

# Configuración de credenciales de Sightengine
API_USER = ''
API_SECRET = ''

# 1. Cargar el archivo de respuestas existente (con los datos de Hive en la pos 0)
#    Si la variable respuestas_modelos ya existe en memoria, se usará directamente.
filename_json = "respuestas_modelos.json"

if 'respuestas_modelos' not in locals() and os.path.exists(filename_json):
    with open(filename_json, "r", encoding="utf-8") as f:
        respuestas_modelos = json.load(f)
    print(f"✓ Archivo '{filename_json}' cargado correctamente.")

if 'dataset_tuple' in locals():
    print(f"Iniciando evaluación con Sightengine para {len(dataset_tuple)} imágenes...\n")
    
    # Parámetros fijos para la API de Sightengine
    params = {
        'models': 'deepfake',
        'api_user': API_USER,
        'api_secret': API_SECRET
    }

    for idx, (clasificacion, tipo_fake, file_path) in enumerate(dataset_tuple):
        print(f"[{idx + 1}/{len(dataset_tuple)}] Procesando Sightengine: {os.path.basename(file_path)} ({tipo_fake})")
        
        try:
            # Enviar el archivo local en el cuerpo de la petición (multipart/form-data)
            with open(file_path, 'rb') as f:
                files = {'media': f}
                
                response = requests.post(
                    'https://api.sightengine.com/1.0/check.json',
                    files=files,
                    data=params
                )
                
                if response.status_code == 200:
                    res_sightengine = response.json()
                else:
                    res_sightengine = {"error_code": response.status_code, "message": response.text}

        except Exception as e:
            res_sightengine = {"error": str(e)}
        
        # Asignar la respuesta en la POSICIÓN 2 (índice 1) de la estructura
        if idx < len(respuestas_modelos):
            respuestas_modelos[idx][1] = res_sightengine
        else:
            # En caso de que no existiera el elemento previo, se crea la estructura [Hive, Sightengine, Modelo3]
            respuestas_modelos.append([None, res_sightengine, None])
            
        # Pausa ligera recomendada
        time.sleep(0.1)

    print(f"\n✓ Proceso finalizado. Se han guardado las respuestas de Sightengine en la posición 2.")

    # Guardar la estructura actualizada en el archivo JSON
    with open(filename_json, "w", encoding="utf-8") as out_file:
        json.dump(respuestas_modelos, out_file, ensure_ascii=False, indent=4)
    print(f"✓ Archivo '{filename_json}' actualizado con éxito.")

else:
    print("❌ Error: La tupla 'dataset_tuple' no existe.")

✓ Archivo 'respuestas_modelos.json' cargado correctamente.
❌ Error: La tupla 'dataset_tuple' no existe.


In [ ]:
import requests
import json
import os
import time

# Configuración correcta de Hive
API_KEY_HIVE = ''
filename_json = "respuestas_modelos.json"
url_hive = 'https://api.thehive.ai/api/v3/hive/ai-generated-and-deepfake-content-detection'

headers_hive = {
    'authorization': f'Bearer {API_KEY_HIVE}'
}

# Cargar el archivo de respuestas existente si existe
if 'respuestas_modelos' not in locals() and os.path.exists(filename_json):
    with open(filename_json, "r", encoding="utf-8") as f:
        respuestas_modelos = json.load(f)
    print(f"✓ Archivo '{filename_json}' cargado correctamente.")

if 'dataset_tuple' in locals():
    print(f"Iniciando evaluación controlada con Hive para {len(dataset_tuple)} imágenes...\n")

    if 'respuestas_modelos' not in locals():
        respuestas_modelos = []

    for idx, (clasificacion, tipo_fake, file_path) in enumerate(dataset_tuple):
        print(f"[{idx + 1}/{len(dataset_tuple)}] Procesando Hive: {os.path.basename(file_path)}")
        
        res_hive = None
        max_retries = 5
        delay = 2.0  # Tiempo base de espera para backoff exponencial

        # Bucle de reintento en caso de error 429
        for intento in range(1, max_retries + 1):
            try:
                with open(file_path, 'rb') as f:
                    files = {'media': (os.path.basename(file_path), f, 'image/jpeg')}
                    response = requests.post(url_hive, headers=headers_hive, files=files)
                
                # Caso 1: Petición Exitosa
                if response.status_code == 200:
                    res_hive = response.json()
                    
                    # Validar si Hive responde 200 pero incluye payload de límites (429)
                    msg = str(res_hive.get('message', ''))
                    if 'usage_limit' in msg or 'rate limit' in msg.lower() or res_hive.get('error_code') == 429:
                        print(f"    [!] Throttling detectado en JSON (Intento {intento}/{max_retries}). Pausando {delay}s...")
                        time.sleep(delay)
                        delay *= 2
                        continue
                    
                    break  # Éxito: salir del bucle de reintentos

                # Caso 2: Bloqueo directo HTTP 429 (Too Many Requests)
                elif response.status_code == 429:
                    print(f"    [!] Error HTTP 429 (Límite de peticiones) en intento {intento}/{max_retries}. Pausando {delay}s...")
                    time.sleep(delay)
                    delay *= 2

                # Caso 3: Otros errores HTTP (400, 401, 403, 500, etc.)
                else:
                    res_hive = {"error_code": response.status_code, "message": response.text}
                    break

            except Exception as e:
                print(f"    [!] Error de conexión en intento {intento}/{max_retries}: {e}. Reintentando en {delay}s...")
                time.sleep(delay)
                delay *= 2

        # Asignar la respuesta en la POSICIÓN 1 (Índice 0)
        if idx < len(respuestas_modelos):
            respuestas_modelos[idx][0] = res_hive
        else:
            respuestas_modelos.append([res_hive, None, None])
            
        # CONTROL DE RITMO PREVENTIVO: Pausa de 1 segundo entre peticiones para evitar saturar la API
        time.sleep(1.0) 

    print(f"\n✓ Proceso finalizado. Se han guardado las respuestas de Hive.")

    # Guardar progreso en archivo local
    with open(filename_json, "w", encoding="utf-8") as out_file:
        json.dump(respuestas_modelos, out_file, ensure_ascii=False, indent=4)
    print(f"✓ Archivo '{filename_json}' actualizado con éxito.")

else:
    print("❌ Error: La tupla 'dataset_tuple' no existe.")

In [ ]:
import requests
import json
import os
import time
import base64

API_KEY = ''

headers = {
    'Authorization': f'Bearer {API_KEY}',
    'Content-Type': 'application/json'
}

filename_json = "respuestas_modelos.json"
url_detector = 'https://app.deepfakedetector.ai/api/v1/detect/image'

def es_error_429(res):
    if not isinstance(res, dict):
        return False
    if res.get('error_code') in [429, 420]:
        return True
    msg = str(res.get('message', '')).lower()
    return 'usage_limit' in msg or 'rate limit' in msg or 'throttled' in msg or 'too many requests' in msg

# 1. Cargar las respuestas guardadas previamente
if 'respuestas_modelos' not in locals() and os.path.exists(filename_json):
    with open(filename_json, "r", encoding="utf-8") as f:
        respuestas_modelos = json.load(f)
    print(f"✓ Archivo '{filename_json}' cargado correctamente.")

if 'dataset_tuple' in locals():
    print(f"Iniciando evaluación controlada con DeepfakeDetector.ai para {len(dataset_tuple)} imágenes...\n")
    
    if 'respuestas_modelos' not in locals():
        respuestas_modelos = []

    # Sincronizar tamaño del array
    while len(respuestas_modelos) < len(dataset_tuple):
        respuestas_modelos.append([None, None, None])

    for idx, (clasificacion, tipo_fake, file_path) in enumerate(dataset_tuple):
        # Asegurar sub-lista de 3 posiciones
        while len(respuestas_modelos[idx]) < 3:
            respuestas_modelos[idx].append(None)

        # OMITIR SI YA FUE PROCESADA CON ÉXITO
        res_previa = respuestas_modelos[idx][2]
        if res_previa is not None and "error_code" not in res_previa and "error" not in res_previa:
            print(f"[{idx + 1}/{len(dataset_tuple)}] ⏭️ Omitiendo DeepfakeDetector (ya procesada): {os.path.basename(file_path)}")
            continue

        print(f"[{idx + 1}/{len(dataset_tuple)}] Procesando DeepfakeDetector: {os.path.basename(file_path)}")
        
        res_detector = None
        max_retries = 5
        delay = 2.0

        # Convertir la imagen local a Base64
        try:
            with open(file_path, 'rb') as image_file:
                encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
            
            # FORMATO EXACTO EXIGIDO POR LA API DE DEEPFAKEDETECTOR.AI
            image_payload = {
                "image_base64": encoded_string
            }
        except Exception as e:
            res_detector = {"error": f"Error al codificar imagen local: {str(e)}"}
            print(f"    [!] Error en Base64: {e}")
            respuestas_modelos[idx][2] = res_detector
            continue

        for intento in range(1, max_retries + 1):
            try:
                response = requests.post(
                    url_detector,
                    headers=headers,
                    json=image_payload
                )

                if response.status_code in [200, 201]:
                    res_detector = response.json()
                    
                    if es_error_429(res_detector):
                        print(f"    [!] Throttling detectado (Intento {intento}/{max_retries}). Pausando {delay}s...")
                        time.sleep(delay)
                        delay *= 2
                        continue

                    break

                elif response.status_code == 429:
                    print(f"    [!] Error HTTP 429 en intento {intento}/{max_retries}. Pausando {delay}s...")
                    time.sleep(delay)
                    delay *= 2

                else:
                    res_detector = {"error_code": response.status_code, "message": response.text}
                    break

            except Exception as e:
                print(f"    [!] Error de conexión en intento {intento}/{max_retries}: {e}. Esperando {delay}s...")
                time.sleep(delay)
                delay *= 2

        # Guardar en la posición 2 (Modelo 3)
        respuestas_modelos[idx][2] = res_detector

        # Guardado incremental en disco
        with open(filename_json, "w", encoding="utf-8") as out_file:
            json.dump(respuestas_modelos, out_file, ensure_ascii=False, indent=4)
            
        time.sleep(1.0)

    print(f"\n✓ Proceso completado. Los resultados han sido guardados en '{filename_json}'.")

else:
    print("❌ Error: La tupla 'dataset_tuple' no existe.")

Iniciando evaluación controlada con DeepfakeDetector.ai para 1000 imágenes...

[1/1000] Procesando DeepfakeDetector: real_001.jpg
[2/1000] Procesando DeepfakeDetector: real_002.jpg
[3/1000] Procesando DeepfakeDetector: real_003.jpg
[4/1000] Procesando DeepfakeDetector: real_004.jpg
[5/1000] Procesando DeepfakeDetector: real_005.jpg
[6/1000] Procesando DeepfakeDetector: real_006.jpg
[7/1000] Procesando DeepfakeDetector: real_007.jpg
[8/1000] Procesando DeepfakeDetector: real_008.jpg
[9/1000] Procesando DeepfakeDetector: real_009.jpg
[10/1000] Procesando DeepfakeDetector: real_010.jpg
[11/1000] Procesando DeepfakeDetector: real_011.jpg
[12/1000] Procesando DeepfakeDetector: real_012.jpg
[13/1000] Procesando DeepfakeDetector: real_013.jpg
[14/1000] Procesando DeepfakeDetector: real_014.jpg
[15/1000] Procesando DeepfakeDetector: real_015.jpg
[16/1000] Procesando DeepfakeDetector: real_016.jpg
[17/1000] Procesando DeepfakeDetector: real_017.jpg
[18/1000] Procesando DeepfakeDetector: real_01